# Chapter 6b — RAG over Research Papers

Companion code for the **second half of Chapter 6** of *Build an Advanced RAG Application (From Scratch)*.

Same retrieve → augment → generate pipeline as 6a, but a very different corpus: 500 research-paper abstracts from the [DBLP-v10 dataset on Kaggle](https://www.kaggle.com/datasets/nechbamohammed/research-papers-dataset). New things in this notebook:

- **Document chunking** — papers can exceed the embedder's context window.
- **Direct `transformers`** instead of `sentence-transformers` — useful when you want pooling control.
- **Metadata filtering** by year — show only post-2010 papers, etc.
- **Persistent Qdrant** — collection survives across notebook restarts.

> Reusable code: `chunking.py`, `qdrant_helpers.py`.

## 0. Setup

You need three keys in `.env`:

- `OPEN_ROUTER_API_KEY` — for generation
- `KAGGLE_USERNAME` and `KAGGLE_KEY` — for downloading the dataset

Get a Kaggle token at https://www.kaggle.com/settings → *Create New Token*.

In [ ]:
import os, sys, csv
sys.path.insert(0, '.')

# Lift the CSV size limit — DBLP rows can be big
csv.field_size_limit(sys.maxsize)

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from pathlib import Path

load_dotenv(Path('..').resolve() / '.env')

## 1. Download the dataset (Kaggle)

`kagglehub` reads `KAGGLE_USERNAME` / `KAGGLE_KEY` from the environment. The dataset is ~150MB.

In [ ]:
import kagglehub

path = kagglehub.dataset_download("nechbamohammed/research-papers-dataset")
print("Path:", path)
print(os.listdir(path))

In [ ]:
df = pd.read_csv(Path(path) / "dblp-v10.csv")
print(df.shape)
df.head()

## 2. Prepare a small slice for the demo

Full DBLP is too big to embed in a notebook session. We take 500 papers, drop those without abstracts, and shape each row into a `{page_content, metadata}` dict.

In [ ]:
df_small = df.head(500).copy()
df_small = df_small.dropna(subset=["abstract"]).reset_index(drop=True)

documents = [
    {
        "page_content": row["abstract"],
        "metadata": {
            "source": row["title"],
            "authors": row["authors"],
            "year": row["year"],
            "venue": row["venue"],
            "paper_id": row["id"],
        },
    }
    for _, row in df_small.iterrows()
]
print(f"Documents: {len(documents)}")

## 3. Chunk the documents

Most abstracts fit comfortably in one chunk, but the same code handles longer papers. See `chunking.py`.

In [ ]:
from chunking import simple_recursive_split

chunks = []
for doc in documents:
    chunks.extend(simple_recursive_split(doc, chunk_size=4000, chunk_overlap=200))

print(f"Total chunks: {len(chunks)}")
print(f"First chunk length: {len(chunks[0]['page_content'])}")
chunks[0]

## 4. Generate embeddings (transformers, mean-pooled)

We load the model with `transformers` directly so we can see exactly how the pooling works.

In [ ]:
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

tokenizer = AutoTokenizer.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
text_model = AutoModel.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True).to(device)

def embed(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        outputs = text_model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).cpu()[0].numpy()

# Smoke test
v = embed("autoassociative neural network with dynamic synapses")
print(f"Embedding dim: {v.shape}")

In [ ]:
embeddings = [embed(c["page_content"]) for c in chunks]
print(f"Embeddings: {len(embeddings)} x {embeddings[0].shape}")

## 5. Index in Qdrant

Same helpers as 6a; different collection name.

In [ ]:
from qdrant_helpers import make_in_memory_client, reset_collection, upsert_text_chunks, query_papers
from qdrant_client import models

qdrant = make_in_memory_client()
COLLECTION = "research_collection"
VEC_SIZE = embeddings[0].shape[0]
reset_collection(qdrant, COLLECTION, vector_size=VEC_SIZE)
upsert_text_chunks(qdrant, COLLECTION, chunks, embeddings)
print(f"Upserted {len(chunks)} chunks into '{COLLECTION}'.")

## 6. Query

In [ ]:
hits = query_papers(
    "an autoassociative neural network with dynamic synapses",
    embed,
    qdrant,
    collection_name=COLLECTION,
    limit=3,
)
for i, h in enumerate(hits, 1):
    md_meta = h.payload["metadata"]
    print(f"{i}. {md_meta['source']}  ({md_meta['year']}, {md_meta['venue']})")
    print(f"   {h.payload['content'][:240]}...\n")

### With a year filter

Qdrant supports server-side filters on payload fields — pass them at query time.

In [ ]:
query_vec = embed("an autoassociative neural network with dynamic synapses")
hits = qdrant.query_points(
    collection_name=COLLECTION,
    query=query_vec.tolist(),
    query_filter=models.Filter(
        must=[models.FieldCondition(key="metadata.year", range=models.Range(gte=2000))]
    ),
    limit=3,
).points

for i, h in enumerate(hits, 1):
    m = h.payload["metadata"]
    print(f"{i}. {m['source']}  (year={m['year']})  score={h.score:.3f}")

## 7. Full RAG over papers

In [ ]:
from openai import OpenAI

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.getenv("OPEN_ROUTER_API_KEY"))

def query_qdrant_for_rag(query, limit=5):
    hits = query_papers(query, embed, qdrant, collection_name=COLLECTION, limit=limit)
    return [
        {
            "source_id": i + 1,
            "content": h.payload["content"],
            "metadata": h.payload["metadata"],
        }
        for i, h in enumerate(hits)
    ]

def generate_paper_answer(query, llm_model="qwen/qwen3-8b"):
    sources = query_qdrant_for_rag(query)
    prompt = f'''Based on the following query, generate a comprehensive answer.
Cite inline as [1][2] and mention authors, paper titles, and venues. Be precise.

Query: "{query}"

Sources:
{sources}

Return in Markdown format.'''
    stream = client.chat.completions.create(
        model=llm_model,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
    )
    out = ""
    for chunk in stream:
        d = chunk.choices[0].delta.content
        if d:
            print(d, end="", flush=True)
            out += d
    print()
    return out, sources

In [ ]:
answer, sources = generate_paper_answer("How do autoassociative neural networks with dynamic synapses store information?")

## Wrap

You now have the same RAG pattern instantiated on two completely different corpora — hotel reviews and research papers. The pieces (embedder, vector DB, retriever, prompt template, LLM) are interchangeable; the architecture is the same.